Question Answering on Google Privacy Policy using RAG + BART

In [ ]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import PyPDF2
import re
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM as ModelLoader, AutoTokenizer as TokenLoader, AutoModelForCausalLM
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pnd
import nltk

# Ensure the Punkt tokenizer for sentence splitting is available
nltk.download('punkt')

# Function to extract text from a PDF document
def get_text_from_pdf(pdf_path):
    pdf_text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            pdf_text += page.extract_text()
    return pdf_text

# Path to the PDF document containing the privacy policy
document_path = 'google_privacy_policy_en.pdf'
document_text = get_text_from_pdf(document_path)

# Break the extracted text into sentences
split_sentences = sent_tokenize(document_text)

# Function to clean and format a sentence
def process_sentence(input_sentence):
    input_sentence = re.sub(r'\[\d+\]', '', input_sentence)  # Remove references in square brackets
    input_sentence = ' '.join(input_sentence.split())       # Eliminate extra spaces
    return input_sentence

processed_sentences = [process_sentence(sentence) for sentence in split_sentences if len(sentence) > 20]

# Group sentences into chunks of three with a step of one sentence
sentence_chunks = []
for index in range(len(processed_sentences) - 2):
    combined_chunk = ' '.join(processed_sentences[index:index + 3])
    sentence_chunks.append(combined_chunk)

# Display the first five generated chunks
print("Sample Chunks:")
for idx, chunk in enumerate(sentence_chunks[:5]):
    print(f"Chunk {idx + 1}: {chunk}")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Sample Chunks:
Chunk 1: Privacy Policy Last modified: December 18, 2017 ( view archived versions ) (The hyperlinked examples are available at the end of this document.) There are many different ways you can use our services – to search for and share information, to communicate with other people or to create new content. When you share information with us, for example by creating a Google Account , we can make those services even better – to show you more relevant search results and ads, to help you connect with people or to make sharing with others quicker and easier .
Chunk 2: There are many different ways you can use our services – to search for and share information, to communicate with other people or to create new content. When you share information with us, for example by creating a Google Account , we can make those services even better – to show you more relevant search results and ads, to help you connect with people or to make sharing with others quicker and easier . As you u

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Initialize Sentence-BERT model for embedding generation
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for the text chunks
text_chunk_embeddings = embedding_model.encode(sentence_chunks, convert_to_tensor=True)
text_chunk_embeddings


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

tensor([[-0.0828, -0.0368, -0.0221,  ...,  0.0787, -0.0416, -0.0083],
        [-0.0803, -0.0428, -0.0113,  ...,  0.0742, -0.0167, -0.0481],
        [-0.1028,  0.0084,  0.0093,  ...,  0.0752, -0.0333, -0.0274],
        ...,
        [-0.0650, -0.0990,  0.0155,  ...,  0.0213, -0.0211,  0.0325],
        [-0.0494, -0.1110, -0.0029,  ...,  0.0469,  0.0054, -0.0193],
        [-0.0578, -0.0281, -0.0418,  ...,  0.0277,  0.0140, -0.0581]])

**Loading BART and defining the RAG function and NON- RAG function for BART**

In [ ]:
# Initialize the BART model for text summarization
bart_model_identifier = "facebook/bart-large-cnn"
bart_text_tokenizer = AutoTokenizer.from_pretrained(bart_model_identifier)
bart_text_model = AutoModelForSeq2SeqLM.from_pretrained(bart_model_identifier)

# Function to generate responses using RAG (Retrieval-Augmented Generation)
def create_response_with_rag(user_query, top_matches=3):
    # Encode the user query
    query_embedding = embedding_model.encode(user_query, convert_to_tensor=True)

    # Calculate similarity between query and chunk embeddings
    similarity_scores = cosine_similarity(
        query_embedding.cpu().numpy().reshape(1, -1),
        text_chunk_embeddings.cpu().numpy()
    )
    # Select top-k most relevant chunks
    best_indices = np.argsort(similarity_scores[0])[-top_matches:][::-1]
    relevant_chunks = [sentence_chunks[index] for index in best_indices]

    # Combine the query with the retrieved chunks for context
    contextual_input = f"Question: {user_query} Context: {' '.join(relevant_chunks)}"

    # Truncate the input if it exceeds model's maximum length
    max_allowed_length = bart_text_model.config.max_position_embeddings  # Typically 1024
    if len(contextual_input) > max_allowed_length:
        contextual_input = contextual_input[:max_allowed_length]

    # Generate a response based on the input
    input_tokens = bart_text_tokenizer(contextual_input, return_tensors="pt", truncation=True)
    model_outputs = bart_text_model.generate(
        **input_tokens,
        max_new_tokens=150,
        num_beams=4,
        early_stopping=True
    )
    final_response = bart_text_tokenizer.decode(model_outputs[0], skip_special_tokens=True)

    return final_response, contextual_input, relevant_chunks


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [ ]:
# Function to generate a response without using RAG (Retrieval-Augmented Generation)
def create_response_without_rag(user_query):
    # Tokenize the input query
    tokenized_input = bart_text_tokenizer(user_query, return_tensors="pt")

    # Generate a response directly from the model
    model_output = bart_text_model.generate(
        **tokenized_input,
        max_new_tokens=150,
        num_beams=4,
        early_stopping=True
    )

    # Decode the model's output into text
    return bart_text_tokenizer.decode(model_output[0], skip_special_tokens=True)


**Evaluating the Questions (Queries)**

In [ ]:
# Define a list of user queries for testing
user_queries = [
    "What data does Google collect from users?",
    "When does Google share user data externally?",
    "What privacy controls are available to users?",
    "How does Google manage cookies and tracking?"
]

# Evaluate and compare responses with and without RAG
response_evaluations = []
for query in user_queries:
    print(f"\nOriginal Question:\n{query}")

    # Generate response without RAG
    print("\nResponse Without RAG:")
    response_no_rag = create_response_without_rag(query)
    print(response_no_rag)

    # Generate response with RAG
    print("\nContextualized Prompt:")
    response_rag, contextualized_input, retrieved_context = create_response_with_rag(query, top_matches=5)
    print(contextualized_input)  # Display the constructed input for RAG

    print("\nResponse With RAG:")
    print(response_rag)  # Display the response generated with context

    # Store the results for analysis
    response_evaluations.append({
        'Question': query,
        'Response_Without_RAG': response_no_rag,
        'Contextualized_Prompt': contextualized_input,
        'Response_With_RAG': response_rag,
        'Retrieved_Context': retrieved_context
    })


Original Question:
What data does Google collect from users?

Response Without RAG:


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


What data does Google collect from users? What do they do with the data they collect? What does Google do with all the data it collects? Do you know what Google does with your data? Share your story with CNN iReport. Share your photos, videos and other content with iReport and we'll feature them in a next story.

Contextualized Prompt:
Question: What data does Google collect from users? Context: For example, an accelerometer can be used to determine things like speed, or a gyroscope to figure out direction of travel. "collect information" This includes information like your usage data and preferences, Gmail messages, G+ profile, photos, videos, browsing history, map searches, docs, or other Google-hosted content. "combine personal information from one service with information, including personal information, from other Google services" For example, when you’re signed in to your Google Account and search on Google, you can see search results from the public web, along with pages, photos

**Loading BART and defining the RAG function and NON- RAG function for Flan-T5 XL**

In [ ]:
# Initialize the Flan-T5 XL model for advanced text generation
flan_t5_identifier = "google/flan-t5-xl"
flan_t5_text_tokenizer = AutoTokenizer.from_pretrained(flan_t5_identifier)
flan_t5_text_model = AutoModelForSeq2SeqLM.from_pretrained(flan_t5_identifier)


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.44k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/53.0k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.45G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
# Function to generate a response with RAG (Retrieval-Augmented Generation) using Flan-T5 XL

def create_response_with_rag_flant5(user_query, top_matches=7):
    # Encode the user query into an embedding
    query_embedding = embedding_model.encode(user_query, convert_to_tensor=True)

    # Compute similarities between the query and text chunk embeddings
    similarity_scores = cosine_similarity(
        query_embedding.cpu().numpy().reshape(1, -1),
        text_chunk_embeddings.cpu().numpy()
    )
    # Identify the top-k most relevant chunks
    best_chunk_indices = np.argsort(similarity_scores[0])[-top_matches:][::-1]
    relevant_text_chunks = [sentence_chunks[idx] for idx in best_chunk_indices]

    # Combine retrieved chunks with the question to provide context
    contextual_text = ' '.join(relevant_text_chunks)
    complete_input = f"Question: {user_query} Context: {contextual_text} Provide a detailed and specific answer."

    # Truncate input if it exceeds the Flan-T5 XL model's maximum token limit
    max_token_length = 512
    if len(complete_input.split()) > max_token_length:
        complete_input = ' '.join(complete_input.split()[:max_token_length])

    # Generate the response
    tokenized_input = flan_t5_text_tokenizer(complete_input, return_tensors="pt", truncation=True)
    model_output = flan_t5_text_model.generate(
        **tokenized_input,
        max_new_tokens=250,
        num_beams=4,
        do_sample=True,
        temperature=1.0,
        early_stopping=True
    )
    generated_response = flan_t5_text_tokenizer.decode(model_output[0], skip_special_tokens=True)

    return generated_response, relevant_text_chunks  # Return both the response and retrieved context

# Function to generate a response without using RAG (Retrieval-Augmented Generation) with Flan-T5 XL
def create_response_without_rag_flant5(user_query):
    # Tokenize the input query
    tokenized_query = flan_t5_text_tokenizer(user_query, return_tensors="pt", truncation=True)

    # Generate a response directly from the model
    model_response = flan_t5_text_model.generate(
        **tokenized_query,
        max_new_tokens=250,
        num_beams=4,
        do_sample=True,
        temperature=1.0,
        early_stopping=True
    )

    # Decode and return the response
    return flan_t5_text_tokenizer.decode(model_response[0], skip_special_tokens=True)


**Evaluating the Questions (Queries)**


In [ ]:
# Define a set of user queries to evaluate
evaluation_queries = [
    "What data does Google collect from users?",
    "When does Google share user data externally?",
    "What privacy controls are available to users?",
    "How does Google manage cookies and tracking?"
]

# Evaluate the responses both with and without RAG
evaluation_results = []
for query in evaluation_queries:
    print(f"\nOriginal Question:\n{query}")

    # Generate response without using RAG
    print("\nResponse Without RAG:")
    response_no_rag = create_response_without_rag_flant5(query)
    print(response_no_rag)

    # Generate response using RAG
    print("\nContextualized Prompt:")
    response_rag, context_chunks = create_response_with_rag_flant5(query, top_matches=5)
    contextualized_prompt = f"Question: {query} Context: {' '.join(context_chunks)}"
    print(contextualized_prompt)

    print("\nResponse With RAG:")
    print(response_rag)

    # Store the evaluation results for further analysis
    evaluation_results.append({
        'Question': query,
        'Response_Without_RAG': response_no_rag,
        'Response_With_RAG': response_rag,
        'Retrieved_Context': context_chunks,
        'Contextualized_Prompt': contextualized_prompt
    })


Original Question:
What data does Google collect from users?

Response Without RAG:
Google collects information about your use of the Google Services, including your IP address, geographical location, browser type, referring website, time spent on site, and other information.

Contextualized Prompt:
Question: What data does Google collect from users? Context: For example, an accelerometer can be used to determine things like speed, or a gyroscope to figure out direction of travel. "collect information" This includes information like your usage data and preferences, Gmail messages, G+ profile, photos, videos, browsing history, map searches, docs, or other Google-hosted content. "combine personal information from one service with information, including personal information, from other Google services" For example, when you’re signed in to your Google Account and search on Google, you can see search results from the public web, along with pages, photos, and Google+ posts from your friends

**Comparative Discussion**

**Key Observations:**
**BART-Large-CNN:**

**Without RAG: Often, the model struggled with providing specific answers, resorting to generic or repetitive outputs. For instance, it included irrelevant or redundant phrases like "Share your story with CNN iReport."**

**With RAG: The model performed well when given appropriate chunks as context, generating detailed and meaningful responses, particularly for questions like:
"What data does Google collect from users?"**

**Flan-T5 XL:**

**Without RAG: Showed better general understanding compared to BART, but responses were still less specific and lacked sufficient depth.**

**With RAG: The model excelled at providing precise, well-structured answers leveraging the context. For instance:**
**"How does Google manage cookies and tracking?"**

**The model's ability to synthesize information across the chunks was superior, providing more coherent answers.**

**Strengths of Flan-T5 XL:
Handles detailed context better due to fine-tuning for reasoning and generative tasks.
Outputs are more natural and aligned with human expectations, even with complex questions.**

**Strengths of BART-Large-CNN:
Performs reasonably well for summarization tasks, as seen in the RAG setting.
Contextual answers are clear when provided with the right context.**

**Weaknesses Noticed:
Both models suffered from vague answers in the Non-RAG setting, primarily due to a lack of domain-specific pretraining.
BART was prone to generating verbose or irrelevant details compared to Flan-T5 XL.**

In [ ]:

# Function to get model summary
def get_model_summary(model_name):
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    config = model.config
    summary = {
        "Model": model_name,
        "Number of Parameters": f"{model.num_parameters():,}",
        "Model Architecture": config.architectures[0] if config.architectures else "Unknown",
        "Max Position Embeddings": getattr(config, 'max_position_embeddings', 'Not Available'),
        "Hidden Size": getattr(config, 'd_model', getattr(config, 'hidden_size', 'Not Available')),
        "Number of Layers": getattr(config, 'num_layers', getattr(config, 'encoder_layers', 'Not Available')),
        "Number of Attention Heads": getattr(config, 'num_attention_heads', 'Not Available'),
    }
    del model
    return summary

# Models to summarize
model_names = ["facebook/bart-large-cnn", "google/flan-t5-xl"]

# Generate model summaries
print("### Model Summaries ###\n")
model_summaries = [get_model_summary(model_name) for model_name in model_names]
for summary in model_summaries:
    print(f"Model: {summary['Model']}")
    for key, value in summary.items():
        if key != "Model":
            print(f"- {key}: {value}")
    print()

# Hyperparameters data
hyperparameters = {
    "Model": ["facebook/bart-large-cnn", "google/flan-t5-xl"],
    "Retriever": ["Sentence-BERT (all-MiniLM-L6-v2)", "Sentence-BERT (all-MiniLM-L6-v2)"],
    "Top-K Chunks": [5, 7],
    "Beam Search": [4, 4],
    "Max Tokens": [150, 250],
    "Truncation Length": ["1024 (BART)", "512 (FLAN-T5)"],
    "Model Size": ["Large", "Extra Large"],
    "Hidden Layers": [12, 24],
    "Optimizer": ["AdamW", "AdamW"],
    "Learning Rate": [5e-5, 5e-5],
}

# Print hyperparameters in a clean format
print("### Simplified Hyperparameters Table ###\n")
print(f"{'Model':<25} {'Retriever':<35} {'Top-K Chunks':<15} {'Beam Search':<15} {'Max Tokens':<15} {'Truncation Length':<20} {'Model Size':<15} {'Hidden Layers':<15} {'Optimizer':<10} {'Learning Rate':<15}")
print("="*165)
for i in range(len(hyperparameters["Model"])):
    print(f"{hyperparameters['Model'][i]:<25} {hyperparameters['Retriever'][i]:<35} {hyperparameters['Top-K Chunks'][i]:<15} {hyperparameters['Beam Search'][i]:<15} {hyperparameters['Max Tokens'][i]:<15} {hyperparameters['Truncation Length'][i]:<20} {hyperparameters['Model Size'][i]:<15} {hyperparameters['Hidden Layers'][i]:<15} {hyperparameters['Optimizer'][i]:<10} {hyperparameters['Learning Rate'][i]:<15}")


### Model Summaries ###



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model: facebook/bart-large-cnn
- Number of Parameters: 406,290,432
- Model Architecture: BartForConditionalGeneration
- Max Position Embeddings: 1024
- Hidden Size: 1024
- Number of Layers: 12
- Number of Attention Heads: 16

Model: google/flan-t5-xl
- Number of Parameters: 2,849,757,184
- Model Architecture: T5ForConditionalGeneration
- Max Position Embeddings: Not Available
- Hidden Size: 2048
- Number of Layers: 24
- Number of Attention Heads: 32

### Simplified Hyperparameters Table ###

Model                     Retriever                           Top-K Chunks    Beam Search     Max Tokens      Truncation Length    Model Size      Hidden Layers   Optimizer  Learning Rate  
facebook/bart-large-cnn   Sentence-BERT (all-MiniLM-L6-v2)    5               4               150             1024 (BART)          Large           12              AdamW      5e-05          
google/flan-t5-xl         Sentence-BERT (all-MiniLM-L6-v2)    7               4               250             512 (FLAN-T5)

In [ ]:
from transformers import AutoModelForSeq2SeqLM as ModelLoader, AutoTokenizer as TokenLoader

# Function to extract model details
def extract_model_details(model_identifier):
    loaded_model = ModelLoader.from_pretrained(model_identifier)
    model_config = loaded_model.config
    details = {
        "Model Name": model_identifier,
        "Parameter Count": f"{loaded_model.num_parameters():,}",
        "Architecture": model_config.architectures[0] if model_config.architectures else "Unknown",
        "Max Position Embeddings": getattr(model_config, 'max_position_embeddings', 'Not Available'),
        "Hidden Layer Size": getattr(model_config, 'd_model', getattr(model_config, 'hidden_size', 'Not Available')),
        "Total Layers": getattr(model_config, 'num_layers', getattr(model_config, 'encoder_layers', 'Not Available')),
        "Attention Heads": getattr(model_config, 'num_attention_heads', 'Not Available'),
    }
    del loaded_model  # Free up memory after loading the model
    return details

# List of models to summarize
models_to_summarize = ["facebook/bart-large-cnn", "google/flan-t5-xl"]

# Print model summaries
print("### Model Overview ###\n")
model_overviews = [extract_model_details(model) for model in models_to_summarize]
for overview in model_overviews:
    print(f"Model Name: {overview['Model Name']}")
    for key, value in overview.items():
        if key != "Model Name":
            print(f"- {key}: {value}")
    print()

# Hyperparameter information
hyperparameter_data = {
    "Model": ["facebook/bart-large-cnn", "google/flan-t5-xl"],
    "Retriever": ["Sentence-BERT (all-MiniLM-L6-v2)", "Sentence-BERT (all-MiniLM-L6-v2)"],
    "Top-K Chunks": [5, 7],
    "Beam Search": [4, 4],
    "Max Tokens": [150, 250],
    "Truncation Length": ["1024 (BART)", "512 (FLAN-T5)"],
    "Model Size": ["Large", "Extra Large"],
    "Hidden Layers": [12, 24],
    "Optimizer": ["AdamW", "AdamW"],
    "Learning Rate": [5e-5, 5e-5],
}

# Display hyperparameters in a formatted table
print("### Hyperparameters Overview ###\n")
print(f"{'Model':<25} {'Retriever':<35} {'Top-K Chunks':<15} {'Beam Search':<15} {'Max Tokens':<15} {'Truncation Length':<20} {'Model Size':<15} {'Hidden Layers':<15} {'Optimizer':<10} {'Learning Rate':<15}")
print("="*165)
for idx in range(len(hyperparameter_data["Model"])):
    print(f"{hyperparameter_data['Model'][idx]:<25} {hyperparameter_data['Retriever'][idx]:<35} {hyperparameter_data['Top-K Chunks'][idx]:<15} {hyperparameter_data['Beam Search'][idx]:<15} {hyperparameter_data['Max Tokens'][idx]:<15} {hyperparameter_data['Truncation Length'][idx]:<20} {hyperparameter_data['Model Size'][idx]:<15} {hyperparameter_data['Hidden Layers'][idx]:<15} {hyperparameter_data['Optimizer'][idx]:<10} {hyperparameter_data['Learning Rate'][idx]:<15}")


### Model Overview ###



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model Name: facebook/bart-large-cnn
- Parameter Count: 406,290,432
- Architecture: BartForConditionalGeneration
- Max Position Embeddings: 1024
- Hidden Layer Size: 1024
- Total Layers: 12
- Attention Heads: 16

Model Name: google/flan-t5-xl
- Parameter Count: 2,849,757,184
- Architecture: T5ForConditionalGeneration
- Max Position Embeddings: Not Available
- Hidden Layer Size: 2048
- Total Layers: 24
- Attention Heads: 32

### Hyperparameters Overview ###

Model                     Retriever                           Top-K Chunks    Beam Search     Max Tokens      Truncation Length    Model Size      Hidden Layers   Optimizer  Learning Rate  
facebook/bart-large-cnn   Sentence-BERT (all-MiniLM-L6-v2)    5               4               150             1024 (BART)          Large           12              AdamW      5e-05          
google/flan-t5-xl         Sentence-BERT (all-MiniLM-L6-v2)    7               4               250             512 (FLAN-T5)        Extra Large     24          